# EGRFC Backend Exploration

Simple notebook for interrogating the canonical backend database (`data/egrfc_backend.duckdb`).

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

candidate_paths = [
    Path.cwd() / "data" / "egrfc_backend.duckdb",
    Path.cwd() / "data" / "egrfc_backend_alt.duckdb",
]

last_error = None
con = None
db_path = None

for candidate in candidate_paths:
    if not candidate.exists():
        continue
    try:
        con = duckdb.connect(str(candidate), read_only=True)
        db_path = candidate
        break
    except Exception as exc:
        last_error = exc

if con is None:
    if last_error:
        raise RuntimeError(f"Could not open any backend DB in read-only mode: {last_error}")
    raise FileNotFoundError("No backend DB found. Build one with python/build_backend.py")

print(f"Using database (read-only): {db_path}")

Using database (read-only): /Users/samlindsay/Documents/Projects/Personal/EGRFC/egrfc-stats/data/egrfc_backend.duckdb


In [2]:
# Quick helper
def q(sql: str, limit: int | None = None):
    df = con.execute(sql).df()
    if limit is not None:
        return df.head(limit)
    return df

In [3]:
# List tables and views
print("Tables and views in the database:")
print(q("SELECT table_name, table_type FROM information_schema.tables WHERE table_schema = 'main';"))

Tables and views in the database:
                                 table_name  table_type
0                                     games  BASE TABLE
1                                 games_rfu  BASE TABLE
2                                  lineouts  BASE TABLE
3              pitchero_appearance_backfill  BASE TABLE
4        pitchero_appearance_reconciliation  BASE TABLE
5                                   players  BASE TABLE
6                        player_appearances  BASE TABLE
7                    player_appearances_rfu  BASE TABLE
8                 player_profiles_canonical  BASE TABLE
9                            season_scorers  BASE TABLE
10                  season_summary_enriched  BASE TABLE
11                                set_piece  BASE TABLE
12                squad_continuity_enriched  BASE TABLE
13         squad_position_profiles_enriched  BASE TABLE
14                     squad_stats_enriched  BASE TABLE
15     squad_stats_with_thresholds_enriched  BASE TABLE
16         v_p

In [7]:
q("SELECT * FROM games LIMIT 5;")

,game_id,squad,date,season,competition,game_type,opposition,home_away,score_for,score_against,result,captain,vice_captain_1,vice_captain_2
0,2017-02-25_1st_Pulborough_2,1st,2017-02-25,2016/17,League,League,Pulborough II,A,31,18,W,Mark Oliver,None,None
1,2017-09-09_1st_Ditchling,1st,2017-09-09,2017/18,League,League,Ditchling,H,50,5,W,None,None,None
2,2017-09-16_1st_Burgess_Hill,1st,2017-09-16,2017/18,League,League,Burgess Hill,H,23,21,W,None,None,None
3,2017-09-23_1st_Barns_Green,1st,2017-09-23,2017/18,League,League,Barns Green,A,61,5,W,None,None,None
4,2017-09-30_1st_Seaford,1st,2017-09-30,2017/18,League,League,Seaford,H,14,6,W,None,None,None


In [8]:

import altair as alt

# Join game-level data with red zone performance metrics
df = q(
"""
SELECT 
    g.game_id, 
    g.date,
    g.season,
    g.game_type,
    g.opposition || ' (' || g.home_away || ')' AS label,
    r.team, 
    r.opposition, 
    r.entries_22m, 
    r.points, 
    r.tries, 
    r.points_per_entry, 
    r.tries_per_entry
FROM v_red_zone r
LEFT JOIN games g USING (game_id)       
"""
)


df_totals = q("""SELECT
    team,
    NULL AS game_id,
    'AVERAGE' AS opposition,
    AVG(entries_22m) AS entries_22m,
    AVG(points) AS points,
    AVG(tries) AS tries,
    AVG(points)/AVG(entries_22m) AS points_per_entry,
    AVG(tries)/AVG(entries_22m) AS tries_per_entry
FROM v_red_zone
GROUP BY team
""")
df

,game_id,date,season,game_type,label,team,opposition,entries_22m,points,tries,points_per_entry,tries_per_entry
0,2024-09-07_1st_London_Cornish,2024-09-07,2024/25,League,London Cornish (H),Opposition,London Cornish,6,18,2,3.0,0.333333
1,2024-12-14_1st_Old_Haileyburians,2024-12-14,2024/25,League,Old Haileyburians (H),Opposition,Old Haileyburians,5,5,1,1.0,0.200000
2,2025-02-15_1st_Eastbourne,2025-02-15,2024/25,League,Eastbourne (H),Opposition,Eastbourne,7,15,2,2.1,0.285714
3,2025-03-22_1st_Cobham,2025-03-22,2024/25,League,Cobham (H),Opposition,Cobham,11,43,7,3.9,0.636364
4,2025-04-12_1st_Haywards_Heath,2025-04-12,2024/25,Cup,Haywards Heath (H),Opposition,Haywards Heath,10,37,6,3.7,0.600000
5,2025-10-04_1st_Seaford,2025-10-04,2025/26,League,Seaford (A),Opposition,Seaford,5,10,2,2.0,0.400000
6,2025-10-11_1st_Shoreham,2025-10-11,2025/26,League,Shoreham (H),Opposition,Shoreham,4,12,2,3.0,0.500000
7,2025-11-08_1st_Lewes,2025-11-08,2025/26,League,Lewes (H),Opposition,Lewes,4,14,2,3.5,0.500000
8,2025-11-15_1st_Hove,2025-11-15,2025/26,Cup,Hove (H),Opposition,Hove,12,32,5,2.7,0.416667
9,2025-11-22_1st_Pulborough,2025-11-22,2025/26,League,Pulborough (A),Opposition,Pulborough,9,12,2,1.3,0.222222


## 1) Table row counts

In [5]:
tables = q("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main';")["table_name"].tolist()
sql = " UNION ALL ".join(f"SELECT '{t}' AS table_name, COUNT(*) AS rows FROM {t}" for t in tables) + " ORDER BY table_name"

print("\nRow counts for each table/view:")
print(q(sql))


Row counts for each table/view:
                                 table_name   rows
0                                     games    273
1                                 games_rfu    709
2                                  lineouts   1235
3              pitchero_appearance_backfill    183
4        pitchero_appearance_reconciliation    430
5                        player_appearances   5030
6                    player_appearances_rfu  15715
7                                   players    280
8                            season_scorers    965
9                                 set_piece    222
10                        v_lineout_summary    311
11         v_pitchero_appearance_mismatches    244
12  v_player_appearance_discrepancy_summary    120
13                        v_player_profiles    342
14                  v_rfu_average_retention    156
15                    v_rfu_lineup_coverage    101
16                    v_rfu_match_retention   2721
17                         v_rfu_squad_size    15

## 2) Season coverage

In [6]:
q("""
SELECT season, squad, COUNT(*) AS games
FROM games
GROUP BY season, squad
ORDER BY squad, season
""")

,season,squad,games
0,2016/17,1st,1
1,2017/18,1st,21
2,2018/19,1st,21
3,2019/20,1st,16
4,2021/22,1st,20
5,2022/23,1st,20
6,2023/24,1st,26
7,2024/25,1st,24
8,2025/26,1st,18
9,2016/17,2nd,6


In [11]:
q("""
SELECT season, squad, COUNT(*) AS appearances
FROM player_appearances
GROUP BY season, squad
ORDER BY squad, season
""")

,season,squad,appearances
0,2016/17,1st,19
1,2017/18,1st,324
2,2018/19,1st,350
3,2019/20,1st,270
4,2021/22,1st,399
5,2022/23,1st,364
6,2023/24,1st,491
7,2024/25,1st,436
8,2025/26,1st,334
9,2016/17,2nd,108


## 3) Integrity checks

In [7]:
# Duplicate keys check
checks = {
    "games_pk_dupes": """
        SELECT squad, date, opposition, COUNT(*) AS c
        FROM games
        GROUP BY squad, date, opposition
        HAVING COUNT(*) > 1
    """,
    "apps_pk_dupes": """
        SELECT squad, date, player, COUNT(*) AS c
        FROM player_appearances
        GROUP BY squad, date, player
        HAVING COUNT(*) > 1
    """,
    "lineouts_pk_dupes": """
        SELECT squad, date, seq_id, COUNT(*) AS c
        FROM lineouts
        GROUP BY squad, date, seq_id
        HAVING COUNT(*) > 1
    """,
}

for name, sql in checks.items():
    df = q(sql)
    print(f"{name}: {len(df)} issue rows")
    if not df.empty:
        display(df.head(20))

games_pk_dupes: 0 issue rows
apps_pk_dupes: 0 issue rows
lineouts_pk_dupes: 0 issue rows


In [8]:
# Sanity checks
q("""
SELECT
  SUM(CASE WHEN score_for IS NULL OR score_against IS NULL THEN 1 ELSE 0 END) AS games_missing_score,
  SUM(CASE WHEN lineouts_total < lineouts_won THEN 1 ELSE 0 END) AS lineout_bad_totals,
  SUM(CASE WHEN scrums_total < scrums_won THEN 1 ELSE 0 END) AS scrum_bad_totals
FROM games
LEFT JOIN set_piece USING (game_id)
""")

,games_missing_score,lineout_bad_totals,scrum_bad_totals
0,0.0,0.0,0.0


## 4) View outputs used by frontend/charts

In [9]:
q("SELECT * FROM v_season_results ORDER BY season DESC, squad, game_type")

,season,squad,game_type,played,won,lost,drawn,points_for,points_against
0,2025/26,1st,Cup,3,2.0,1.0,0.0,94.0,77.0
1,2025/26,1st,Friendly,1,0.0,1.0,0.0,19.0,24.0
2,2025/26,1st,League,14,12.0,2.0,0.0,512.0,191.0
3,2025/26,2nd,Cup,2,1.0,1.0,0.0,63.0,86.0
4,2025/26,2nd,League,9,5.0,4.0,0.0,177.0,199.0
5,2024/25,1st,Cup,2,1.0,1.0,0.0,61.0,65.0
6,2024/25,1st,League,22,3.0,18.0,1.0,380.0,717.0
7,2024/25,2nd,Cup,1,0.0,1.0,0.0,22.0,31.0
8,2024/25,2nd,Friendly,6,4.0,2.0,0.0,154.0,104.0
9,2024/25,2nd,League,13,7.0,6.0,0.0,317.0,308.0


In [22]:
# Lineout success by area
zone = q(
    """
    SELECT 
        CASE WHEN area == 'Front' THEN 1 WHEN area == 'Middle' THEN 2 WHEN area == 'Back' THEN 3 END AS zone, 
        SUM(total) AS total, 
        SUM(won) AS won, 
        ROUND(SUM(won) * 100.0 / NULLIF(SUM(total), 0), 1) / 100.0 AS pct_won 
        FROM v_lineout_summary 
        GROUP BY zone 
        ORDER BY pct_won DESC
    """, 
    limit=100)

jumper_sql = """
    SELECT 
        'Jumper' AS player_type,
        jumper AS player, 
        SUM(total) AS total, 
        SUM(won) AS won, 
        ROUND(SUM(won) * 100.0 / NULLIF(SUM(total), 0), 1) / 100.0 AS pct_won, 
        SQRT((SUM(won)/SUM(total) * (1 - SUM(won)/SUM(total)))/SUM(total)) AS error_margin,
        SUM(total * zone)/SUM(total) AS avg_zone
    FROM (SELECT *, CASE WHEN area == 'Front' THEN 1 WHEN area == 'Middle' THEN 2 WHEN area == 'Back' THEN 3 END AS zone FROM v_lineout_summary) AS subquery
    WHERE squad = '1st'
    GROUP BY jumper
    HAVING SUM(total) >= 20
    ORDER BY pct_won DESC
    """

thrower_sql = """
    SELECT 
        'Thrower' AS player_type,
        thrower AS player, 
        SUM(total) AS total, 
        SUM(won) AS won, 
        ROUND(SUM(won) * 100.0 / NULLIF(SUM(total), 0), 1) / 100.0 AS pct_won, 
        SQRT((SUM(won)/SUM(total) * (1 - SUM(won)/SUM(total)))/SUM(total)) AS error_margin,
        SUM(total * zone)/SUM(total) AS avg_zone
    FROM (SELECT *, CASE WHEN area == 'Front' THEN 1 WHEN area == 'Middle' THEN 2 WHEN area == 'Back' THEN 3 END AS zone FROM v_lineout_summary) AS subquery
    WHERE squad = '1st'
    GROUP BY thrower
    HAVING SUM(total) >= 20
    ORDER BY pct_won DESC
    """

jumpers = q(jumper_sql, limit=100)

throwers = q(thrower_sql, limit=100)

players = q(f"({jumper_sql}) UNION ALL ({thrower_sql})")

players

,player_type,player,total,won,pct_won,error_margin,avg_zone
0,Jumper,Oscar Staples,21.0,19.0,0.905,0.064056,1.809524
1,Jumper,Rory Evans,40.0,35.0,0.875,0.052291,1.575000
2,Jumper,John Peaty,231.0,193.0,0.835,0.024392,1.562771
3,Jumper,Sam Lindsay-McCall,388.0,303.0,0.781,0.020998,2.244845
4,Jumper,Ryland Thomas,95.0,68.0,0.716,0.046275,1.421053
5,Jumper,Tom Mooney,72.0,49.0,0.681,0.054949,1.972222
6,Jumper,Dan Billin,24.0,13.0,0.542,0.101707,2.000000
7,Thrower,Mark Oliver,39.0,31.0,0.795,0.064659,1.820513
8,Thrower,Ben Tottman,775.0,611.0,0.788,0.014672,1.874839
9,Thrower,George Beet,63.0,38.0,0.603,0.061638,1.984127


In [23]:

# Plot success (y-axis) line by zone (x-axis), with scatter points for jumpers by average zone
import altair as alt
from python.chart_helpers import *

size_scale = alt.Scale(domain=[10,1000], range=[10,400])

base = alt.Chart(zone).encode(
    x=alt.X(
        "zone:Q", 
        title="Average Lineout Zone", 
        axis=alt.Axis(tickCount=3, values=[1, 2, 3], labelExpr="datum.value == 1 ? 'Front' : datum.value == 2 ? 'Middle' : 'Back'", grid=False, ticks=False, labelFontSize=16), 
        scale=alt.Scale(domain=[0.75, 3.25], nice=False)
    ),
    y=alt.Y("pct_won:Q", title="Success Rate", scale=alt.Scale(domain=[0.5, 1.0]), axis=alt.Axis(format="%", tickCount=5)),
)   

avg_points = base.encode(size=alt.Size("total:Q", legend=None, scale=size_scale)).mark_point(color="#991515", fill="red", fillOpacity=0.9)
avg_line = base.mark_line(color="#991515")
avg_total_labels = avg_points.encode(text="total:Q", size=alt.value(12)).mark_text(align="right", dx=-10, dy=5, opacity=0.5, color="#991515")

points = alt.Chart(players).encode(
    x=alt.X("avg_zone:Q", title="Average Lineout Zone"),
    y=alt.Y("pct_won:Q", title="Success Rate"),
    size=alt.Size("total:Q", legend=None, scale=size_scale),
    color=alt.Color("player_type:N", scale=alt.Scale(domain=["Thrower", "Jumper"], range=["#146f14","#202946"]), title=None,legend=alt.Legend(orient="top-right")),
    tooltip=["player_type", "player", "total", "won", "pct_won", "avg_zone"],
)

errorbars = points.mark_errorbar(opacity=0.5).encode(
    x="avg_zone:Q",
    y=alt.Y("ymin:Q", scale=alt.Scale(clamp=True), title="Success Rate"),
    y2=alt.Y2("ymax:Q", title=None)
).transform_calculate(
    ymin="datum.pct_won - datum.error_margin",
    ymax="datum.pct_won + datum.error_margin"
)

# Add text labels for jumpers
labels = points.encode(text="player:N", size=alt.value(12)).mark_text(align="left", dx=10)
total_labels = points.encode(
    text="total:Q",
    size=alt.value(12), 
    color=alt.value("black")
).mark_text(align="right", dx=-10, opacity=0.5)

chart = (
    avg_line
    + avg_points
    + avg_total_labels
    + errorbars 
    + points.mark_circle(size=100, opacity=1., fillOpacity=0.9) 
    + labels
    + total_labels
).resolve_scale(
    y="shared", x="shared"
).properties(
    title=alt.Title(
        text="Lineout Success by Zone", 
        subtitle=[
            "1st XV average success rate since 2021 (red), with individual players by their average zone.",
            "Size and numbers indicate total lineouts taken. Error bars show uncertainty based on sample size."
        ]
    ),
    width=500,
    height=600
)
chart#.save("lineout_success_by_zone.html")

alt.LayerChart(...)

In [89]:
wide

team,squad,set_piece_type,season,EGRFC,Opposition,EGRFC_lineouts,Opposition_lineouts,season_pos
0,1st XV,Lineout,2021/22,0.903226,0.659341,93.0,91.0,0
1,1st XV,Lineout,2022/23,0.847458,0.736559,177.0,186.0,1
2,1st XV,Lineout,2023/24,0.697595,0.799163,291.0,239.0,2
3,1st XV,Lineout,2024/25,0.723502,0.804878,217.0,205.0,3
4,1st XV,Lineout,2025/26,0.833333,0.711409,144.0,149.0,4
5,1st XV,Scrum,2021/22,0.910714,0.901639,93.0,91.0,0
6,1st XV,Scrum,2022/23,0.944056,0.876923,177.0,186.0,1
7,1st XV,Scrum,2023/24,0.919355,0.766871,291.0,239.0,2
8,1st XV,Scrum,2024/25,0.980263,0.808333,217.0,205.0,3
9,1st XV,Scrum,2025/26,0.966942,0.758929,144.0,149.0,4


## 5) Player deep dive

## 5) Player deep dive

In [4]:
player_name = "Max Crawley-Moore"  # change as needed

print("Profile")
display(q(f"SELECT * FROM players WHERE name = '{player_name}'"))

print("Appearances by season")
display(q(f"""
    SELECT season, squad, COUNT(*) AS apps, SUM(CASE WHEN is_starter THEN 1 ELSE 0 END) AS starts
    FROM player_appearances
    WHERE player = '{player_name}'
    GROUP BY season, squad
    ORDER BY squad, season
"""))

print("Scoring by season")
display(q(f"""
    SELECT season, squad, tries, conversions, penalties, drop_goals, points
    FROM season_scorers
    WHERE player = '{player_name}'
    ORDER BY squad, season
"""))

print("Selections by position")
display(q(f"""
    SELECT position, COUNT(*) AS selections
    FROM player_appearances
    WHERE player = '{player_name}'
    GROUP BY position
    ORDER BY position
"""))

Profile


,name,short_name,position,squad,first_appearance_date,first_appearance_squad,first_appearance_opposition,photo_url,sponsor,total_appearances,total_starts,total_captaincies,total_vc_appointments,total_lineouts_jumped,lineouts_won_as_jumper,career_points
0,Max Crawley-Moore,Max C,Centre,1st,1900-01-01,2nd,Burgess Hill II,img/headshots/MaxCrawleyMoore.png,None,120,105,38,4,0,0,209


Appearances by season


,season,squad,apps,starts
0,2017/18,1st,10,7.0
1,2018/19,1st,3,2.0
2,2019/20,1st,13,9.0
3,2021/22,1st,19,19.0
4,2022/23,1st,18,18.0
5,2023/24,1st,16,14.0
6,2024/25,1st,10,9.0
7,2025/26,1st,10,8.0
8,2017/18,2nd,6,6.0
9,2018/19,2nd,2,0.0


Scoring by season


,season,squad,tries,conversions,penalties,drop_goals,points
0,2017/18,1st,1,0,0,0,5
1,2018/19,1st,0,0,0,0,0
2,2019/20,1st,10,0,0,0,50
3,2021/22,1st,6,0,0,0,30
4,2022/23,1st,5,1,0,0,27
5,2023/24,1st,2,0,0,0,10
6,2024/25,1st,1,0,0,0,5
7,2025/26,1st,3,0,0,0,15
8,2017/18,2nd,6,0,0,0,30
9,2018/19,2nd,1,0,0,0,5


Selections by position


,position,selections
0,Bench,10
1,Centre,104
2,Wing,1
3,None,5


## 6) Team/opposition drilldown

In [16]:
opposition = "Hove"  # change as needed
q(f"""
SELECT date, season, squad, home_away, score_for, score_against, result, game_type
FROM games
WHERE opposition = '{opposition}'
ORDER BY date DESC
""")

,date,season,squad,home_away,score_for,score_against,result,game_type
0,2026-02-14,2025/26,2nd,H,7,47,L,League
1,2025-11-15,2025/26,1st,H,15,32,L,Cup
2,2025-11-08,2025/26,2nd,A,19,59,L,League
3,2025-02-08,2024/25,1st,A,11,18,L,League
4,2024-10-12,2024/25,1st,H,21,34,L,League
5,2024-03-23,2023/24,1st,H,13,15,L,League
6,2023-12-09,2023/24,1st,A,26,12,W,League
7,2023-03-04,2022/23,2nd,A,24,57,L,League
8,2023-01-07,2022/23,2nd,H,43,19,W,League
9,2022-05-07,2021/22,2nd,A,29,20,W,Cup


# Set Piece

In [5]:
q(f"SELECT * FROM set_piece WHERE entries_22m > 0")

,squad,date,team,lineouts_won,lineouts_total,lineouts_success_rate,scrums_won,scrums_total,scrums_success_rate,entries_22m,points_per_22m_entry,tries_per_22m_entry,game_id,season,opposition


In [4]:
q("SELECT * FROM set_piece WHERE entries_22m > 0")

,squad,date,team,lineouts_won,lineouts_total,lineouts_success_rate,scrums_won,scrums_total,scrums_success_rate,entries_22m,points_per_22m_entry,tries_per_22m_entry,game_id,season,opposition


In [16]:
q("SELECT * FROM lineouts")

,squad,date,seq_id,half,numbers,call,call_type,dummy,area,drive,crusaders,transfer,flyby,thrower,jumper,won,game_id,season,opposition
0,1st,2021-08-28,1,1,5,,Other,False,Middle,False,False,False,False,Ben Tottman,Sam Lindsay-McCall,True,2021-08-28_1st_London_Irish,2021/22,London Irish
1,1st,2021-08-28,2,2,5,RD,Old,True,Middle,False,False,False,False,Ben Tottman,Sam Lindsay-McCall,True,2021-08-28_1st_London_Irish,2021/22,London Irish
2,1st,2021-08-28,3,2,6,Red,Old,False,Front,False,False,False,False,Ben Tottman,Gus Nye,True,2021-08-28_1st_London_Irish,2021/22,London Irish
3,1st,2021-08-28,4,1,4,Snap,4-man only,False,Front,False,False,False,False,Ben Tottman,Gus Nye,True,2021-08-28_1st_London_Irish,2021/22,London Irish
4,1st,2021-08-28,5,1,6,Even,Old,False,Middle,True,False,False,False,Ben Tottman,Sam Lindsay-McCall,True,2021-08-28_1st_London_Irish,2021/22,London Irish
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1230,2nd,2026-03-07,2,1,5,11-,Other,False,Middle,True,False,False,False,Reggie Harris,Tim Morris,True,2026-03-07_2nd_Hellingly,2025/26,Hellingly
1231,2nd,2026-03-07,3,1,5,2+,Other,False,Middle,False,False,False,False,Reggie Harris,Tim Morris,True,2026-03-07_2nd_Hellingly,2025/26,Hellingly
1232,2nd,2026-03-07,4,2,5,1,SLM,False,Front,True,False,False,False,Reggie Harris,Greg Brack,True,2026-03-07_2nd_Hellingly,2025/26,Hellingly
1233,2nd,2026-03-07,5,2,5,1,SLM,False,Front,False,False,False,False,Reggie Harris,Greg Brack,True,2026-03-07_2nd_Hellingly,2025/26,Hellingly


In [ ]:
# Close connection when done
con.close()

In [4]:
import requests
from bs4 import BeautifulSoup

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0 Safari/537.36"
})

squad_label = "1st"
team_id = "142068"
season_id = "47693"
            
fixtures_url = f"https://www.egrfc.com/teams/{team_id}/fixtures-results?season={season_id}"


response = session.get(fixtures_url, timeout=20)
response.raise_for_status()
fixtures_soup = BeautifulSoup(response.content, "html.parser")


In [5]:
fixtures_soup

<!DOCTYPE html>
<html lang="en" style="width:100%;overflow-x:hidden"><head><meta charset="utf-8"/><meta content="initial-scale=1.0, width=device-width" name="viewport"/><title>EG Men 1 fixtures &amp; results</title><link href="https://img-res.pitchero.com" rel="preconnect"/><link href="https://www.googletagmanager.com" rel="preconnect"/><link href="https://www.google-analytics.com" rel="preconnect"/><style>@font-face{font-display:swap;font-family:Anton;font-style:normal;font-weight:400;src:local("Anton Regular"),local("Anton-Regular"),url(/fonts/anton-regular.woff2) format('woff2');}@font-face{font-display:swap;font-family:Roboto Condensed;font-style:normal;font-weight:700;src:local("Roboto Condensed Bold"),local("RobotoCondensed-Bold"),url(/fonts/roboto-condensed-v18-latin-700.woff2) format('woff2');}@font-face{font-display:swap;font-family:Montserrat;font-style:normal;font-weight:500;src:local("Montserrat Medium"),local("Montserrat-Medium"),url(/fonts/montserrat-v14-latin-500.woff2) 